In [1]:
from pymilvus import MilvusClient

In [2]:
client = MilvusClient("./RAG.db")

/opt/anaconda3/envs/iaair2/lib/python3.11/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('jordyvl/scibert_scivocab_uncased_sentence_transformer')

def create_embedding(chunks):
    embeddings = model.encode(chunks)
    return embeddings

In [4]:
import requests
import time

def getPaper(paperId="649def34f8be52c8b66281af98ae884c09aef38b"):
    
    url = f"http://api.semanticscholar.org/graph/v1/paper/{paperId}"
    
    query_params = {"fields": "title,year,abstract,citationCount,citations,isOpenAccess,openAccessPdf,venue,fieldsOfStudy"}
    
    api_key = "AWz2yi9RLuOWs21sQdDh2PPF31E41c18GUbQZ5Ie"
    
    headers = {"x-api-key": api_key}
    
    response = requests.get(url, params=query_params, headers=headers)

    if response.status_code == 200:
       response_data = response.json()
       return response_data
    else:
       print(f"Request failed with status code {response.status_code}: {response.text}")

In [26]:
query = "What evaluation metrics are commonly reported in biomedical NLP tasks?"

search_res = client.search(
    collection_name="ingestion_v0",
    data=create_embedding([query]),
    limit=70,  # Return top 3 results
    output_fields=["id", "paperId"],
)

In [29]:
ground_truth_dataset = client.get(
    collection_name = "ingestion_v0",
    ids = [373, 387, 294, 361, 89, 453, 344, 345, 63, 171, 114, 79, 328, 124, 116, 296, 180],
)

In [30]:
ground_truth_dataset_ids = [i['paperId'] for i in ground_truth_dataset]

In [31]:
final_papers = []
for i in search_res[0]:
    if i['paperId'] in ground_truth_dataset_ids:
        final_papers.append(i)

In [25]:
for i in search_res[0]:
    paper = getPaper(i['paperId'])
    print(paper['title'])
    print(paper['abstract'])
    sleep(1)

Assessment of contextualised representations in detecting outcome phrases in clinical trials
Automating the recognition of outcomes reported in clinical trials using machine learning has a huge potential of speeding up access to evidence necessary in healthcare decision-making. Prior research has however acknowledged inadequate training corpora as a challenge for the Outcome detection (OD) task. Additionally, several contextualized representations like BERT and ELMO have achieved unparalleled success in detecting various diseases, genes, proteins, and chemicals, however, the same cannot be emphatically stated for outcomes, because these models have been relatively under-tested and studied for the OD task. We introduce"EBM-COMET", a dataset in which 300 PubMed abstracts are expertly annotated for clinical outcomes. Unlike prior related datasets that use arbitrary outcome classifications, we use labels from a taxonomy recently published to standardize outcome classifications. To extract 

In [17]:
from time import sleep

In [33]:
for i in final_papers:
    paper = getPaper(i['entity']['paperId'])
    print(paper['title'], paper['abstract'])
    sleep(1)

Improving Health Question Answering with Reliable and Time-Aware Evidence Retrieval In today's digital world, seeking answers to health questions on the Internet is a common practice. However, existing question answering (QA) systems often rely on using pre-selected and annotated evidence documents, thus making them inadequate for addressing novel questions. Our study focuses on the open-domain QA setting, where the key challenge is to first uncover relevant evidence in large knowledge bases. By utilizing the common retrieve-then-read QA pipeline and PubMed as a trustworthy collection of medical research documents, we answer health questions from three diverse datasets. We modify different retrieval settings to observe their influence on the QA pipeline's performance, including the number of retrieved documents, sentence selection process, the publication year of articles, and their number of citations. Our results reveal that cutting down on the amount of retrieved documents and favor

In [34]:
client.close()

In [6]:
results = client.query(
    collection_name="ingestion_v0",
    filter="id >= 0",
    output_fields=["*"]
)


In [7]:
results[-1]

{'id': 3367,
 'vector': [],
 'paperId': 'b17969990b3745a494f8fcadae6c8cd0426dc3ec'}

In [17]:
from random import sample

nums = sample(range(0, 3368), 30)  # 20 unique ints from 1–100



In [18]:
nums

[1311,
 2623,
 153,
 3048,
 1477,
 3096,
 1940,
 302,
 1321,
 561,
 547,
 1461,
 1606,
 2105,
 2653,
 3169,
 149,
 3250,
 1362,
 2409,
 109,
 3100,
 426,
 2018,
 3315,
 1605,
 1244,
 72,
 3050,
 2846]

In [19]:
ids = [1, 5, 42, 99]

results = client.get(
    collection_name="ingestion_v0",
    ids=nums
)


In [37]:
#papers = []
paperIds = []

for i in results:
    #paper = getPaper(i['paperId'])
    paperIds.append(i['paperId'])
    #papers.append((paper['title'], paper['abstract']))
    #time.sleep(1)

paperIds = list(set(paperIds))

In [38]:
paperIds

['f1f774dd883260504f02763aa40776ccce23af80',
 '911fe48fc5b6ca5589b48521660db436065cf9c0',
 '192041215f6a1573fbbbbe323ee4275157fb6a3d',
 '686d9ee744fa013cc21cdd86acd864c936e9e456',
 '3a66e3a6fe1f9e1d95140f0c8fefc4ff964ba89d',
 '1e3ef48abeef882e12f9553a1baf8944f3782c88',
 'f00ebe66750aa1996f78b30b8bb77cbf92fab085',
 '2c77f4bfc7b2d5fba5b4d07f337805b41e84e684',
 'b17969990b3745a494f8fcadae6c8cd0426dc3ec',
 '6c7abf617c0591be96a09776092271c72f49ff2f',
 '6ffc29d76c8c31988b5563ac08942bcf28de7bf3',
 '4a497123488f5790548406bd10384ae7b89c4315',
 '428194865ce5732c3c53f548b3d56485e8c7f4eb',
 '1a37f43db8108af003bc9af181460ddaacf9e815',
 '32382cfa1e662de9ccee317ba11e1117b5c99bf7',
 '89abfcec27ae484c368c25b0a1441aaa3bf6e408',
 '61313651e06abb093dfa3fd4430a1f5b146d5b35',
 '5d1b955e38073fc13c82ba46801e300dc1a70a9b',
 '1712eaa82a2e3d081538a17e06416330fac0a0d8',
 'f11512e1213e756a4ed319e11fb4518b9fa17ccb',
 '054dd061a42086ab2758141655bebd7559b144fa',
 '0db5207510819b9956849eb84bfe8703f8f3688d',
 'a60f8498

In [14]:
import time

In [32]:
papers = list(set(papers))

In [33]:
papers

[('Scholarly knowledge graphs through structuring scholarly communication: a review',
  'The necessity for scholarly knowledge mining and management has grown significantly as academic literature and its linkages to authors produce enormously. Information extraction, ontology matching, and accessing academic components with relations have become more critical than ever. Therefore, with the advancement of scientific literature, scholarly knowledge graphs have become critical to various applications where semantics can impart meanings to concepts. The objective of study is to report a literature review regarding knowledge graph construction, refinement and utilization in scholarly domain. Based on scholarly literature, the study presents a complete assessment of current state-of-the-art techniques. We presented an analytical methodology to investigate the existing status of scholarly knowledge graphs (SKG) by structuring scholarly communication. This review paper investigates the field o

In [34]:
for i in papers:
    print(str(i[0])+'\n\n\n\n'+str(i[1])+'\n\n\n\n\n\n\n\n\n\n\n\n\n\n')

Scholarly knowledge graphs through structuring scholarly communication: a review



The necessity for scholarly knowledge mining and management has grown significantly as academic literature and its linkages to authors produce enormously. Information extraction, ontology matching, and accessing academic components with relations have become more critical than ever. Therefore, with the advancement of scientific literature, scholarly knowledge graphs have become critical to various applications where semantics can impart meanings to concepts. The objective of study is to report a literature review regarding knowledge graph construction, refinement and utilization in scholarly domain. Based on scholarly literature, the study presents a complete assessment of current state-of-the-art techniques. We presented an analytical methodology to investigate the existing status of scholarly knowledge graphs (SKG) by structuring scholarly communication. This review paper investigates the field of app

In [39]:
client.close()